# Limpieza de Datos y Generación de 13 Gráficos para Revisión Sistemática

Este notebook contiene el procesamiento, limpieza y graficación interactiva de la base de datos `data3.xlsx` para tu revisión sistemática. Los gráficos generados están diseñados con estándares profesionales para publicaciones científicas, excluyendo marcas de datos no especificados y agregando gráficos de dispersión (scatter plots) sobre el rendimiento temporal.

## 1. Configuración de Librerías y Estilos Visuales

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Estilo científico limpio
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans', 'sans-serif'],
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.titlesize': 16,
    'legend.fontsize': 11,
    'figure.dpi': 150
})

# Paleta de colores personalizada
colors_palette = ['#1D3557', '#457B9D', '#2A9D8F', '#E76F51', '#F4A261', '#E9C46A', '#8D99AE', '#D90429']
sns.set_palette(sns.color_palette(colors_palette))

# Carpeta de salida
os.makedirs('graficos', exist_ok=True)

## 2. Cargar la Base de Datos y Renombrar las Columnas

Cargamos la base de datos `data3.xlsx` y renombramos las columnas a términos limpios en español.

In [ ]:
df = pd.read_excel('data3.xlsx')

clean_cols = [
    'ID', 'Titulo', 'Autores', 'Año', 'DOI', 'Introduccion', 'Metodologia',
    'Analisis_Requerimientos', 'Entorno_Experimental', 'Tamaño_Dataset',
    'Metricas_Rendimiento', 'Limitaciones', 'Discusiones', 'Implementacion_Software',
    'Conclusion', 'Referencia_BibTeX'
]
df.columns = clean_cols
df['Año'] = pd.to_numeric(df['Año'], errors='coerce')
print(f"Base de datos cargada. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

## 3. Extracción de Rendimiento (Accuracy %) y Datos de Publicación (Venues)

In [ ]:
# Extracción de Accuracy numérica mediante Regex
def extraer_accuracy(text):
    if pd.isna(text):
        return None
    text_lower = str(text).lower()
    m1 = re.search(r'accuracy\s+(?:of|reached|is|was)?\s*(\d+(?:\.\d+)?)', text_lower)
    if m1:
        val = float(m1.group(1))
        if val <= 1.0: val *= 100
        return val
    m2 = re.search(r'(\d+(?:\.\d+)?)\s*%\s*accuracy', text_lower)
    if m2:
        val = float(m2.group(1))
        if val <= 1.0: val *= 100
        return val
    m3 = re.search(r'(\d+(?:\.\d+)?)\s*accuracy', text_lower)
    if m3:
        val = float(m3.group(1))
        if val <= 1.0: val *= 100
        return val
    if 'accuracy' in text_lower:
        m4 = re.search(r'(\d+(?:\.\d+)?)', text_lower)
        if m4:
            val = float(m4.group(1))
            if val <= 1.0: val *= 100
            return val
    return None

df['Accuracy_Num'] = df['Metricas_Rendimiento'].apply(extraer_accuracy)

# Tipo de Publicación
def clasificar_bibtex(text):
    if pd.isna(text):
        return 'Otros'
    text_lower = str(text).lower().strip()
    if '@article' in text_lower:
        return 'Revista Científica (Journal)'
    elif '@inproceedings' in text_lower or '@conference' in text_lower:
        return 'Conferencia (Conference)'
    else:
        return 'Otros'

df['Tipo_Publicacion'] = df['Referencia_BibTeX'].apply(clasificar_bibtex)

# Nombre del Venue (Journal / Booktitle)
def extraer_venue(bib):
    if pd.isna(bib):
        return None
    bib_str = str(bib)
    m1 = re.search(r'journal\s*=\s*\{([^\}]+)\}', bib_str, re.IGNORECASE)
    if m1:
        return m1.group(1).strip()
    m2 = re.search(r'booktitle\s*=\s*\{([^\}]+)\}', bib_str, re.IGNORECASE)
    if m2:
        return m2.group(1).strip()
    return None

df['Venue'] = df['Referencia_BibTeX'].apply(extraer_venue)
print("Campos Accuracy, Tipo_Publicacion y Venue extraídos satisfactoriamente.")

## 4. Limpieza de Datos e Inicialización de Tipos

Reemplazamos las celdas vacías y las marcas de datos ausentes (`No especificado`, `nan`, `...`) por valores nulos de Pandas (`pd.NA`).

In [ ]:
for col in df.columns:
    if col not in ['Año', 'Accuracy_Num']:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({'No especificado': pd.NA, 'nan': pd.NA, '<na>': pd.NA, '...': pd.NA})
        df[col] = df[col].astype('string')

print("Limpieza completada. Celdas vacías e indeterminadas representadas por NaN.")

## 5. Clasificaciones Temáticas

Clasificamos el texto libre de `Metodologia` y `Tamaño_Dataset` en categorías amplias.

In [ ]:
# Clasificación de la Metodología
def clasificar_metodologia(text):
    if pd.isna(text):
        return 'Sin especificar'
    text_lower = str(text).lower().strip()
    if text_lower in ['', 'nan', '<na>']:
        return 'Sin especificar'
        
    if any(k in text_lower for k in ['revisión', 'revision', 'review', 'survey', 'literatura', 'estado del arte']):
        return 'Revisión / Estado del Arte'
        
    has_dl = any(k in text_lower for k in [
        'dl', 'deep learning', 'cnn', 'yolo', 'resnet', 'lstm', 'transformer', 'rnn', 
        'gcn', 'ssd', 'neural network', 'redes neuronales', 'mobilenet', 'vgg', 
        'efficientnet', 'bert', 'gpt', 'llm', 'vit', 'attention', 'densenet', 'unet',
        'clip', 'reinforcement learning', 'rl', 'actor-critic', 'deep rl'
    ])
    
    has_ml = any(k in text_lower for k in [
        'ml', 'machine learning', 'svm', 'random forest', 'knn', 'bayesian', 'regres', 
        'pca', 'fp-growth', 'kalman', 'gradient boosting', 'xgboost', 
        'clustering', 'kmeans', 'decision tree', 'árboles de decisión', 'naive bayes',
        'svr', 'gmm', 'plsr'
    ])
    
    if has_dl and has_ml:
        return 'Híbrido (ML + DL)'
    elif has_dl:
        return 'Deep Learning (DL)'
    elif has_ml:
        return 'Machine Learning (ML)'
    elif any(k in text_lower for k in ['opencv', 'dlib', 'visión por computadora', 'computacion edge', 'computer vision', 'image processing', 'procesamiento de imágenes']):
        return 'Visión por Computadora Tradicional'
    else:
        return 'Otros / No especificado'

df['Metodologia_Categorizada'] = df['Metodologia'].apply(clasificar_metodologia)

# Clasificación por Tipo de Dataset
def clasificar_tipo_dataset(text):
    if pd.isna(text):
        return 'No especificado'
    text_lower = str(text).lower().strip()
    if text_lower in ['', 'nan', '<na>', 'no especificado']:
        return 'No especificado'
    
    is_img = any(k in text_lower for k in ['image', 'images', 'imágenes', 'imagenes', 'fotos', 'photo'])
    is_vid = any(k in text_lower for k in ['video', 'videos', 'hours', 'horas', 'frames', 'frames;', 'sec', 'seconds', 'minutos', 'minutes'])
    is_sub = any(k in text_lower for k in ['subject', 'subjects', 'participant', 'participants', 'sujetos', 'participantes', 'personas'])
    
    types = []
    if is_img: types.append('Imágenes')
    if is_vid: types.append('Video / Frames')
    if is_sub: types.append('Sujetos Humanos')
    
    if not types:
        return 'Otros / Especificado'
    return ' & '.join(types)

df['Dataset_Tipo'] = df['Tamaño_Dataset'].apply(clasificar_tipo_dataset)

# --- NUEVOS AGRUPAMIENTOS PARA TABLAS DE CONTINGENCIA (Cruces de Datos 2x2 y 3x3) ---
def simplificar_metodologia_2x2(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras Metodologías'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    return 'Otras Metodologías'

def simplificar_metodologia_3x3(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras / Híbrido / Tradicional'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    if 'Machine Learning' in str(met):
        return 'Machine Learning (ML)'
    return 'Otras / Híbrido / Tradicional'

def agrupar_ano_3x3(yr):
    if pd.isna(yr):
        return 'No especificado'
    yr = int(yr)
    if yr <= 2022:
        return '2020 - 2022'
    elif yr <= 2024:
        return '2023 - 2024'
    else:
        return '2025 - 2026'

def agrupar_entorno_3x3(env):
    if pd.isna(env) or env == 'No especificado':
        return None
    env_lower = str(env).lower()
    if 'cctv' in env_lower or 'videovigilancia' in env_lower:
        return 'Cámaras (CCTV)'
    elif 'drone' in env_lower or 'uav' in env_lower or 'aére' in env_lower or 'aere' in env_lower:
        return 'Drones / UAV'
    elif 'vía pública' in env_lower or 'real' in env_lower or 'calle' in env_lower:
        return 'Vía pública / Real'
    return None

df['Metodologia_2x2'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_2x2)
df['Metodologia_3x3'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_3x3)
df['Periodo_Año'] = df['Año'].apply(agrupar_ano_3x3)
df['Entorno_3x3'] = df['Entorno_Experimental'].apply(agrupar_entorno_3x3)

# Exportar datos limpios a datos_limpios.xlsx (Incluyendo Accuracy extraído, Tipo_Publicacion, Venue y columnas de contingencia)
print("Exportando datos limpios a datos_limpios.xlsx...")

# --- NUEVOS AGRUPAMIENTOS PARA TABLAS DE CONTINGENCIA (Cruces de Datos 2x2 y 3x3) ---
def simplificar_metodologia_2x2(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras Metodologías'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    return 'Otras Metodologías'

def simplificar_metodologia_3x3(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras / Híbrido / Tradicional'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    if 'Machine Learning' in str(met):
        return 'Machine Learning (ML)'
    return 'Otras / Híbrido / Tradicional'

def agrupar_ano_3x3(yr):
    if pd.isna(yr):
        return 'No especificado'
    yr = int(yr)
    if yr <= 2022:
        return '2020 - 2022'
    elif yr <= 2024:
        return '2023 - 2024'
    else:
        return '2025 - 2026'

def agrupar_entorno_3x3(env):
    if pd.isna(env) or env == 'No especificado':
        return None
    env_lower = str(env).lower()
    if 'cctv' in env_lower or 'videovigilancia' in env_lower:
        return 'Cámaras (CCTV)'
    elif 'drone' in env_lower or 'uav' in env_lower or 'aére' in env_lower or 'aere' in env_lower:
        return 'Drones / UAV'
    elif 'vía pública' in env_lower or 'real' in env_lower or 'calle' in env_lower:
        return 'Vía pública / Real'
    return None

df['Metodologia_2x2'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_2x2)
df['Metodologia_3x3'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_3x3)
df['Periodo_Año'] = df['Año'].apply(agrupar_ano_3x3)
df['Entorno_3x3'] = df['Entorno_Experimental'].apply(agrupar_entorno_3x3)

# Exportar datos limpios a datos_limpios.xlsx (Incluyendo Accuracy extraído, Tipo_Publicacion, Venue y columnas de contingencia)
print("Exportando datos limpios a datos_limpios.xlsx...")

# --- NUEVOS AGRUPAMIENTOS PARA TABLAS DE CONTINGENCIA (Cruces de Datos 2x2 y 3x3) ---
def simplificar_metodologia_2x2(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras Metodologías'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    return 'Otras Metodologías'

def simplificar_metodologia_3x3(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras / Híbrido / Tradicional'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    if 'Machine Learning' in str(met):
        return 'Machine Learning (ML)'
    return 'Otras / Híbrido / Tradicional'

def agrupar_ano_3x3(yr):
    if pd.isna(yr):
        return 'No especificado'
    yr = int(yr)
    if yr <= 2022:
        return '2020 - 2022'
    elif yr <= 2024:
        return '2023 - 2024'
    else:
        return '2025 - 2026'

def agrupar_entorno_3x3(env):
    if pd.isna(env) or env == 'No especificado':
        return None
    env_lower = str(env).lower()
    if 'cctv' in env_lower or 'videovigilancia' in env_lower:
        return 'Cámaras (CCTV)'
    elif 'drone' in env_lower or 'uav' in env_lower or 'aére' in env_lower or 'aere' in env_lower:
        return 'Drones / UAV'
    elif 'vía pública' in env_lower or 'real' in env_lower or 'calle' in env_lower:
        return 'Vía pública / Real'
    return None

def agrupar_implementacion_cruce(impl):
    if pd.isna(impl) or impl == 'No especificado':
        return None
    impl_lower = str(impl).lower()
    if 'interfaz' in impl_lower or 'aplicación' in impl_lower or 'aplicacion' in impl_lower or 'software' in impl_lower:
        return 'Aplicación / Interfaz'
    elif 'edge' in impl_lower or 'jetson' in impl_lower or 'raspberry' in impl_lower:
        return 'Edge Computing'
    elif 'drone' in impl_lower or 'uav' in impl_lower:
        return 'Drones / UAV'
    elif 'cctv' in impl_lower or 'cámara' in impl_lower or 'camara' in impl_lower:
        return 'Cámaras / CCTV'
    return None

def agrupar_limitaciones_cruce(lim):
    if pd.isna(lim) or lim == 'No especificado':
        return None
    lim_lower = str(lim).lower()
    if 'iluminación' in lim_lower or 'iluminacion' in lim_lower or 'luz' in lim_lower or 'nocturna' in lim_lower:
        return 'Condiciones de iluminación'
    elif 'costo' in lim_lower or 'recurso' in lim_lower or 'computacional' in lim_lower:
        return 'Costo computacional / Recursos'
    elif 'privacidad' in lim_lower:
        return 'Restricciones de privacidad'
    elif 'desequilibrio' in lim_lower or 'balance' in lim_lower or 'sesgo' in lim_lower:
        return 'Desequilibrio de datasets'
    elif 'oclusión' in lim_lower or 'oclusion' in lim_lower or 'superposición' in lim_lower or 'superposicion' in lim_lower:
        return 'Oclusiones / Superposición'
    return None

df['Metodologia_2x2'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_2x2)
df['Metodologia_3x3'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_3x3)
df['Periodo_Año'] = df['Año'].apply(agrupar_ano_3x3)
df['Entorno_3x3'] = df['Entorno_Experimental'].apply(agrupar_entorno_3x3)
df['Implementacion_Cruce'] = df['Implementacion_Software'].apply(agrupar_implementacion_cruce)
df['Limitacion_Cruce'] = df['Limitaciones'].apply(agrupar_limitaciones_cruce)

# Exportar datos limpios a datos_limpios.xlsx
print("Exportando datos limpios a datos_limpios.xlsx...")

# --- NUEVOS AGRUPAMIENTOS PARA TABLAS DE CONTINGENCIA (Cruces de Datos 2x2 y 3x3) ---
def simplificar_metodologia_2x2(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras Metodologías'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    return 'Otras Metodologías'

def simplificar_metodologia_3x3(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras / Híbrido / Tradicional'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    if 'Machine Learning' in str(met):
        return 'Machine Learning (ML)'
    return 'Otras / Híbrido / Tradicional'

def agrupar_ano_3x3(yr):
    if pd.isna(yr):
        return 'No especificado'
    yr = int(yr)
    if yr <= 2022:
        return '2020 - 2022'
    elif yr <= 2024:
        return '2023 - 2024'
    else:
        return '2025 - 2026'

def agrupar_entorno_3x3(env):
    if pd.isna(env) or env == 'No especificado':
        return 'No especificado'
    env_lower = str(env).lower()
    if 'cctv' in env_lower or 'videovigilancia' in env_lower:
        return 'Cámaras (CCTV)'
    elif 'drone' in env_lower or 'uav' in env_lower or 'aére' in env_lower or 'aere' in env_lower:
        return 'Drones / UAV'
    elif 'vía pública' in env_lower or 'real' in env_lower or 'calle' in env_lower:
        return 'Vía pública / Real'
    return 'Otros'

def agrupar_implementacion_cruce(impl):
    if pd.isna(impl) or impl == 'No especificado':
        return 'No especificado'
    impl_lower = str(impl).lower()
    if 'interfaz' in impl_lower or 'aplicación' in impl_lower or 'aplicacion' in impl_lower or 'software' in impl_lower:
        return 'Aplicación / Interfaz'
    elif 'edge' in impl_lower or 'jetson' in impl_lower or 'raspberry' in impl_lower:
        return 'Edge Computing'
    elif 'drone' in impl_lower or 'uav' in impl_lower:
        return 'Drones / UAV'
    elif 'cctv' in impl_lower or 'cámara' in impl_lower or 'camara' in impl_lower:
        return 'Cámaras / CCTV'
    return 'Otros'

def agrupar_limitaciones_cruce(lim):
    if pd.isna(lim) or lim == 'No especificado':
        return 'No especificado'
    lim_lower = str(lim).lower()
    if 'iluminación' in lim_lower or 'iluminacion' in lim_lower or 'luz' in lim_lower or 'nocturna' in lim_lower:
        return 'Condiciones de iluminación'
    elif 'costo' in lim_lower or 'recurso' in lim_lower or 'computacional' in lim_lower:
        return 'Costo computacional / Recursos'
    elif 'privacidad' in lim_lower:
        return 'Restricciones de privacidad'
    elif 'desequilibrio' in lim_lower or 'balance' in lim_lower or 'sesgo' in lim_lower:
        return 'Desequilibrio de datasets'
    elif 'oclusión' in lim_lower or 'oclusion' in lim_lower or 'superposición' in lim_lower or 'superposicion' in lim_lower:
        return 'Oclusiones / Superposición'
    return 'Otros'

df['Metodologia_2x2'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_2x2)
df['Metodologia_3x3'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_3x3)
df['Periodo_Año'] = df['Año'].apply(agrupar_ano_3x3)
df['Entorno_3x3'] = df['Entorno_Experimental'].apply(agrupar_entorno_3x3)
df['Implementacion_Cruce'] = df['Implementacion_Software'].apply(agrupar_implementacion_cruce)
df['Limitacion_Cruce'] = df['Limitaciones'].apply(agrupar_limitaciones_cruce)

# Exportar datos limpios a datos_limpios.xlsx
print("Exportando datos limpios a datos_limpios.xlsx...")

# --- NUEVOS AGRUPAMIENTOS PARA TABLAS DE CONTINGENCIA (Cruces de Datos 2x2 y 3x3) ---
def simplificar_metodologia_2x2(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras Metodologías'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    return 'Otras Metodologías'

def simplificar_metodologia_3x3(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras / Híbrido / Tradicional'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    if 'Machine Learning' in str(met):
        return 'Machine Learning (ML)'
    return 'Otras / Híbrido / Tradicional'

def agrupar_ano_3x3(yr):
    if pd.isna(yr):
        return 'No especificado'
    yr = int(yr)
    if yr <= 2022:
        return '2020 - 2022'
    elif yr <= 2024:
        return '2023 - 2024'
    else:
        return '2025 - 2026'

def agrupar_entorno_3x3(env):
    if pd.isna(env) or env == 'No especificado':
        return 'No especificado'
    env_lower = str(env).lower()
    if 'cctv' in env_lower or 'videovigilancia' in env_lower:
        return 'Cámaras (CCTV)'
    elif 'drone' in env_lower or 'uav' in env_lower or 'aére' in env_lower or 'aere' in env_lower:
        return 'Drones / UAV'
    elif 'vía pública' in env_lower or 'real' in env_lower or 'calle' in env_lower:
        return 'Vía pública / Real'
    return 'Otros'

def agrupar_implementacion_cruce(impl):
    if pd.isna(impl) or impl == 'No especificado':
        return 'No especificado'
    impl_lower = str(impl).lower()
    if 'interfaz' in impl_lower or 'aplicación' in impl_lower or 'aplicacion' in impl_lower or 'software' in impl_lower:
        return 'Aplicación / Interfaz'
    elif 'edge' in impl_lower or 'jetson' in impl_lower or 'raspberry' in impl_lower:
        return 'Edge Computing'
    elif 'drone' in impl_lower or 'uav' in impl_lower:
        return 'Drones / UAV'
    elif 'cctv' in impl_lower or 'cámara' in impl_lower or 'camara' in impl_lower:
        return 'Cámaras / CCTV'
    return 'Otros'

def agrupar_limitaciones_cruce(lim):
    if pd.isna(lim) or lim == 'No especificado':
        return 'No especificado'
    lim_lower = str(lim).lower()
    if 'iluminación' in lim_lower or 'iluminacion' in lim_lower or 'luz' in lim_lower or 'nocturna' in lim_lower:
        return 'Condiciones de iluminación'
    elif 'costo' in lim_lower or 'recurso' in lim_lower or 'computacional' in lim_lower:
        return 'Costo computacional / Recursos'
    elif 'privacidad' in lim_lower:
        return 'Restricciones de privacidad'
    elif 'desequilibrio' in lim_lower or 'balance' in lim_lower or 'sesgo' in lim_lower:
        return 'Desequilibrio de datasets'
    elif 'oclusión' in lim_lower or 'oclusion' in lim_lower or 'superposición' in lim_lower or 'superposicion' in lim_lower:
        return 'Oclusiones / Superposición'
    return 'Otros'

df['Metodologia_2x2'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_2x2)
df['Metodologia_3x3'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_3x3)
df['Periodo_Año'] = df['Año'].apply(agrupar_ano_3x3)
df['Entorno_3x3'] = df['Entorno_Experimental'].apply(agrupar_entorno_3x3)
df['Implementacion_Cruce'] = df['Implementacion_Software'].apply(agrupar_implementacion_cruce)
df['Limitacion_Cruce'] = df['Limitaciones'].apply(agrupar_limitaciones_cruce)

# Exportar datos limpios a datos_limpios.xlsx
print("Exportando datos limpios a datos_limpios.xlsx...")

# --- NUEVOS AGRUPAMIENTOS PARA TABLAS DE CONTINGENCIA (Cruces de Datos 2x2 y 3x3) ---
def simplificar_metodologia_2x2(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras Metodologías'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    return 'Otras Metodologías'

def simplificar_metodologia_3x3(met):
    if pd.isna(met) or met in ['Sin especificar', 'Otros / No especificado']:
        return 'Otras / Híbrido / Tradicional'
    if 'Deep Learning' in str(met):
        return 'Deep Learning (DL)'
    if 'Machine Learning' in str(met):
        return 'Machine Learning (ML)'
    return 'Otras / Híbrido / Tradicional'

def agrupar_ano_3x3(yr):
    if pd.isna(yr):
        return 'No especificado'
    yr = int(yr)
    if yr <= 2022:
        return '2020 - 2022'
    elif yr <= 2024:
        return '2023 - 2024'
    else:
        return '2025 - 2026'

def agrupar_entorno_3x3(env):
    if pd.isna(env) or env == 'No especificado':
        return 'No especificado'
    env_lower = str(env).lower()
    if 'cctv' in env_lower or 'videovigilancia' in env_lower:
        return 'Cámaras (CCTV)'
    elif 'drone' in env_lower or 'uav' in env_lower or 'aére' in env_lower or 'aere' in env_lower:
        return 'Drones / UAV'
    elif 'vía pública' in env_lower or 'real' in env_lower or 'calle' in env_lower:
        return 'Vía pública / Real'
    return 'Otros'

def agrupar_implementacion_cruce(impl):
    if pd.isna(impl) or impl == 'No especificado':
        return 'No especificado'
    impl_lower = str(impl).lower()
    if 'interfaz' in impl_lower or 'aplicación' in impl_lower or 'aplicacion' in impl_lower or 'software' in impl_lower:
        return 'Aplicación / Interfaz'
    elif 'edge' in impl_lower or 'jetson' in impl_lower or 'raspberry' in impl_lower:
        return 'Edge Computing'
    elif 'drone' in impl_lower or 'uav' in impl_lower:
        return 'Drones / UAV'
    elif 'cctv' in impl_lower or 'cámara' in impl_lower or 'camara' in impl_lower:
        return 'Cámaras / CCTV'
    return 'Otros'

def agrupar_limitaciones_cruce(lim):
    if pd.isna(lim) or lim == 'No especificado':
        return 'No especificado'
    lim_lower = str(lim).lower()
    if 'iluminación' in lim_lower or 'iluminacion' in lim_lower or 'luz' in lim_lower or 'nocturna' in lim_lower:
        return 'Condiciones de iluminación'
    elif 'costo' in lim_lower or 'recurso' in lim_lower or 'computacional' in lim_lower:
        return 'Costo computacional / Recursos'
    elif 'privacidad' in lim_lower:
        return 'Restricciones de privacidad'
    elif 'desequilibrio' in lim_lower or 'balance' in lim_lower or 'sesgo' in lim_lower:
        return 'Desequilibrio de datasets'
    elif 'oclusión' in lim_lower or 'oclusion' in lim_lower or 'superposición' in lim_lower or 'superposicion' in lim_lower:
        return 'Oclusiones / Superposición'
    return 'Otros'

df['Metodologia_2x2'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_2x2)
df['Metodologia_3x3'] = df['Metodologia_Categorizada'].apply(simplificar_metodologia_3x3)
df['Periodo_Año'] = df['Año'].apply(agrupar_ano_3x3)
df['Entorno_3x3'] = df['Entorno_Experimental'].apply(agrupar_entorno_3x3)
df['Implementacion_Cruce'] = df['Implementacion_Software'].apply(agrupar_implementacion_cruce)
df['Limitacion_Cruce'] = df['Limitaciones'].apply(agrupar_limitaciones_cruce)

# Exportar datos limpios a datos_limpios.xlsx
print("Exportando datos limpios a datos_limpios.xlsx...")
df.to_excel('datos_limpios.xlsx', index=False)
print("¡Base de datos exportada con éxito!")


## 6. Generación de los 13 Gráficos (Excluyendo 'Sin Especificar')

In [ ]:
# DEFINICIÓN DE FUNCIONES DE TRAZADO (Soportan gráficos individuales y rejillas)
# ==============================================================================

# ---- GRÁFICO 1: Evolución Temporal de las Publicaciones ----
def plot_publicaciones_por_año(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5.5))
        is_standalone = True
    else:
        is_standalone = False
        
    years_counts = df['Año'].dropna().astype(int).value_counts().sort_index()
    barplot1 = sns.barplot(x=years_counts.index, y=years_counts.values, color='#457B9D', edgecolor='black', linewidth=0.7, ax=ax)
    ax.set_title('Evolución Temporal de las Publicaciones por Año (2020 - 2026)', pad=15, fontweight='bold')
    ax.set_xlabel('Año de Publicación')
    ax.set_ylabel('Cantidad de Artículos')
    
    for p in barplot1.patches:
        height = p.get_height()
        if not pd.isna(height) and height > 0:
            ax.annotate(f'{int(height)}', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                        textcoords='offset points')
                        
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/01_publicaciones_por_año.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 2: Distribución de Metodologías ----
def plot_distribucion_metodologias(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5.5))
        is_standalone = True
    else:
        is_standalone = False
        
    method_counts = df['Metodologia_Categorizada'].value_counts()
    
    barplot2 = sns.barplot(x=method_counts.values, y=method_counts.index, hue=method_counts.index, palette='viridis', edgecolor='black', linewidth=0.7, legend=False, ax=ax)
    ax.set_title('Clasificación General de Metodologías Tecnológicas', pad=15, fontweight='bold')
    ax.set_xlabel('Cantidad de Artículos')
    ax.set_ylabel('Categoría Metodológica')
    
    total_specified_methods = method_counts.sum()
    for p in barplot2.patches:
        val = int(p.get_width())
        pct = (val / total_specified_methods) * 100 if total_specified_methods > 0 else 0
        ax.annotate(f' {val} ({pct:.1f}%)', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    ax.set_xlim(0, max(method_counts.values) * 1.15 if len(method_counts) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/02_distribucion_metodologias.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 3: Entornos Experimentales de Evaluación ----
def plot_entornos_experimentales(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5.5))
        is_standalone = True
    else:
        is_standalone = False
        
    exp_list = []
    for val in df['Entorno_Experimental'].dropna():
        parts = [p.strip() for p in val.split(',')]
        exp_list.extend(parts)
    exp_counts = pd.Series(exp_list).value_counts()
    
    barplot3 = sns.barplot(x=exp_counts.values, y=exp_counts.index, hue=exp_counts.index, palette='crest', edgecolor='black', linewidth=0.7, legend=False, ax=ax)
    ax.set_title('Distribución de Entornos Experimentales de Evaluación', pad=15, fontweight='bold')
    ax.set_xlabel('Número de Menciones')
    ax.set_ylabel('Entorno de Evaluación')
    
    for p in barplot3.patches:
        val = int(p.get_width())
        ax.annotate(f' {val}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    ax.set_xlim(0, max(exp_counts.values) * 1.1 if len(exp_counts) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/03_entornos_experimentales.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 4: Plataformas de Implementación de Software ----
def plot_implementacion_software(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 6))
        is_standalone = True
    else:
        is_standalone = False
        
    impl_list = []
    for val in df['Implementacion_Software'].dropna():
        parts = [p.strip() for p in val.split(',')]
        impl_list.extend(parts)
    impl_counts = pd.Series(impl_list).value_counts()
    impl_counts = impl_counts.head(12)
    
    barplot4 = sns.barplot(x=impl_counts.values, y=impl_counts.index, hue=impl_counts.index, palette='flare', edgecolor='black', linewidth=0.7, legend=False, ax=ax)
    ax.set_title('Plataformas de Implementación / Despliegue de Software (Top 12)', pad=15, fontweight='bold')
    ax.set_xlabel('Número de Menciones')
    ax.set_ylabel('Librería / Hardware / Plataforma')
    
    for p in barplot4.patches:
        val = int(p.get_width())
        ax.annotate(f' {val}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    ax.set_xlim(0, max(impl_counts.values) * 1.1 if len(impl_counts) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/04_implementacion_software.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 5: Limitaciones y Falsos Positivos ----
def plot_limitaciones(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 6))
        is_standalone = True
    else:
        is_standalone = False
        
    lim_list = []
    for val in df['Limitaciones'].dropna():
        parts = [p.strip() for p in val.split(',')]
        lim_list.extend(parts)
    lim_counts = pd.Series(lim_list).value_counts()
    lim_counts = lim_counts.head(10)
    
    barplot5 = sns.barplot(x=lim_counts.values, y=lim_counts.index, hue=lim_counts.index, palette='copper', edgecolor='black', linewidth=0.7, legend=False, ax=ax)
    ax.set_title('Principales Limitaciones y Fuentes de Falsos Positivos (Top 10)', pad=15, fontweight='bold')
    ax.set_xlabel('Número de Menciones')
    ax.set_ylabel('Tipo de Limitación')
    
    for p in barplot5.patches:
        val = int(p.get_width())
        ax.annotate(f' {val}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    ax.set_xlim(0, max(lim_counts.values) * 1.1 if len(lim_counts) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/05_limitaciones.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 6: Métricas de Rendimiento Reportadas ----
def plot_metricas_rendimiento(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
        is_standalone = True
    else:
        is_standalone = False
        
    metrics_dict = {
        'Accuracy / Exactitud': df['Metricas_Rendimiento'].str.lower().str.contains('accuracy|exactitud', na=False).sum(),
        'Precision / Precisión': df['Metricas_Rendimiento'].str.lower().str.contains('precision|precisión', na=False).sum(),
        'Recall / Sensibilidad': df['Metricas_Rendimiento'].str.lower().str.contains('recall|sensibilidad', na=False).sum(),
        'F1-Score': df['Metricas_Rendimiento'].str.lower().str.contains('f1|f-score|f score', na=False).sum(),
        'mAP (Mean Average Precision)': df['Metricas_Rendimiento'].str.lower().str.contains('map', na=False).sum(),
        'FPS (Frames Per Second)': df['Metricas_Rendimiento'].str.lower().str.contains('fps|frames per second|frames/s|cuadros', na=False).sum()
    }
    metrics_series = pd.Series(metrics_dict).sort_values(ascending=False)
    
    barplot6 = sns.barplot(x=metrics_series.values, y=metrics_series.index, hue=metrics_series.index, palette='mako', edgecolor='black', linewidth=0.7, legend=False, ax=ax)
    ax.set_title('Métricas de Rendimiento más Reportadas en la Literatura', pad=15, fontweight='bold')
    ax.set_xlabel('Número de Publicaciones que la Reportan')
    ax.set_ylabel('Métrica')
    
    total_papers = len(df)
    for p in barplot6.patches:
        val = int(p.get_width())
        pct = (val / total_papers) * 100 if total_papers > 0 else 0
        ax.annotate(f' {val} ({pct:.1f}%)', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    ax.set_xlim(0, max(metrics_series.values) * 1.15 if len(metrics_series) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/06_metricas_rendimiento.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 7: Distribución por Tipo de Dataset ----
def plot_tipo_dataset(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
        is_standalone = True
    else:
        is_standalone = False
        
    dataset_counts = df['Dataset_Tipo'].value_counts()
    
    ax.pie(dataset_counts.values, labels=dataset_counts.index, autopct='%1.1f%%', startangle=140, 
           colors=['#1D3557', '#457B9D', '#2A9D8F', '#E76F51', '#8D99AE'][:len(dataset_counts)],
           wedgeprops={'edgecolor': 'black', 'linewidth': 0.8, 'antialiased': True})
    centre_circle = plt.Circle((0,0),0.70,fc='white',edgecolor='black',linewidth=0.5)
    ax.add_artist(centre_circle)
    ax.set_title('Clasificación por Tipo de Dataset Utilizado', pad=20, fontweight='bold')
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/07_tipo_dataset.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 8: Evolución Histórica de Metodologías ----
def plot_evolucion_metodologias(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6.5))
        is_standalone = True
    else:
        is_standalone = False
        
    df_years = df[df['Año'].notna()]
    cross_tab = pd.crosstab(df_years['Año'].astype(int), df_years['Metodologia_Categorizada'])
    cols_order = [c for c in ['Deep Learning (DL)', 'Híbrido (ML + DL)', 'Machine Learning (ML)', 'Revisión / Estado del Arte', 'Visión por Computadora Tradicional', 'Sin especificar', 'Otros / No especificado'] if c in cross_tab.columns]
    cross_tab = cross_tab[cols_order]
    
    cross_tab.plot(kind='bar', stacked=True, color=colors_palette[:len(cols_order)], edgecolor='black', linewidth=0.7, ax=ax)
    ax.set_title('Evolución de Metodologías por Año (2020 - 2026)', pad=15, fontweight='bold')
    ax.set_xlabel('Año de Publicación')
    ax.set_ylabel('Cantidad de Artículos')
    ax.tick_params(axis='x', rotation=0)
    
    if is_standalone:
        ax.legend(title='Metodología', bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.tight_layout()
        plt.savefig('graficos/08_evolucion_metodologias.png', dpi=300, bbox_inches='tight')
        plt.close()
    else:
        ax.legend(title='Metodología', loc='upper left', fontsize=8, title_fontsize=9)

# ---- GRÁFICO 9: Popularidad de las Versiones de YOLO ----
def plot_popularidad_yolo(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5.5))
        is_standalone = True
    else:
        is_standalone = False
        
    yolo_versions = {
        'YOLOv8': df['Metodologia'].str.lower().str.contains('yolov8|yolo v8', na=False).sum(),
        'YOLOv5': df['Metodologia'].str.lower().str.contains('yolov5|yolo v5', na=False).sum(),
        'YOLOv3': df['Metodologia'].str.lower().str.contains('yolov3|yolo v3', na=False).sum(),
        'YOLOv4': df['Metodologia'].str.lower().str.contains('yolov4|yolo v4', na=False).sum(),
        'YOLOv10': df['Metodologia'].str.lower().str.contains('yolov10|yolo v10', na=False).sum(),
        'YOLOv11': df['Metodologia'].str.lower().str.contains('yolov11|yolo v11', na=False).sum(),
        'YOLOv6': df['Metodologia'].str.lower().str.contains('yolov6|yolo v6', na=False).sum(),
        'YOLOv7': df['Metodologia'].str.lower().str.contains('yolov7|yolo v7', na=False).sum(),
        'YOLO Genérico': (df['Metodologia'].str.lower().str.contains('yolo', na=False) & ~df['Metodologia'].str.lower().str.contains('v3|v4|v5|v6|v7|v8|v10|v11', na=False)).sum()
    }
    yolo_series = pd.Series(yolo_versions)
    yolo_series = yolo_series[yolo_series > 0].sort_values(ascending=False)
    
    barplot9 = sns.barplot(x=yolo_series.index, y=yolo_series.values, hue=yolo_series.index, palette='rocket', edgecolor='black', linewidth=0.7, legend=False, ax=ax)
    ax.set_title('Uso y Popularidad de Versiones YOLO en la Literatura', pad=15, fontweight='bold')
    ax.set_xlabel('Versión de YOLO')
    ax.set_ylabel('Cantidad de Publicaciones')
    
    for p in barplot9.patches:
        val = int(p.get_height())
        ax.annotate(f'{val}', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                    textcoords='offset points')
                    
    if not is_standalone:
        ax.tick_params(axis='x', rotation=15)
        
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/09_popularidad_yolo.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 10: Mapa de Calor (Heatmap) Entorno Experimental vs Implementación ----
def plot_heatmap_relacion(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6.5))
        is_standalone = True
    else:
        is_standalone = False
        
    envs = {
        'Cámaras de videovigilancia (CCTV)': 'CCTV',
        'Imágenes aéreas (Drones / UAV)': 'Drones / UAV',
        'Entorno real / Vía pública': 'Vía pública',
        'Simulador de conducción': 'Simulador',
        'No especificado': 'No especificado'
    }
    impls = {
        'Aplicación de Software / Interfaz gráfica': 'App / Interfaz',
        'Edge Computing': 'Edge Computing',
        'Drones / UAV': 'Drones / UAV',
        'Sistemas de Cámaras / CCTV': 'Cámaras / CCTV',
        'Librería OpenCV': 'OpenCV',
        'NVIDIA Jetson': 'NVIDIA Jetson',
        'Raspberry Pi': 'Raspberry Pi',
        'No especificado': 'No especificado'
    }
    
    co_matrix = pd.DataFrame(0, index=envs.values(), columns=impls.values())
    
    for idx, row in df.iterrows():
        row_env = str(row['Entorno_Experimental'])
        row_impl = str(row['Implementacion_Software'])
        if pd.isna(row['Entorno_Experimental']) or pd.isna(row['Implementacion_Software']):
            continue
        for full_env, short_env in envs.items():
            if full_env in row_env:
                for full_impl, short_impl in impls.items():
                    if full_impl in row_impl:
                        co_matrix.loc[short_env, short_impl] += 1
                        
    sns.heatmap(co_matrix, annot=True, cmap="YlGnBu", fmt="d", cbar=True, linewidths=0.5, linecolor='gray',
                cbar_kws={'label': 'Número de Publicaciones Co-incidentes'}, ax=ax)
    ax.set_title('Relación: Entorno Experimental vs. Plataforma de Software', pad=20, fontweight='bold')
    ax.set_xlabel('Plataforma / Hardware de Implementación')
    ax.set_ylabel('Entorno de Prueba Experimental')
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/10_heatmap_relacion.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 11: Evolución de Accuracy (%) por Año (Dispersión con Jitter) ----
def plot_dispersion_accuracy(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 6))
        is_standalone = True
    else:
        is_standalone = False
        
    df_acc = df[df['Accuracy_Num'].notna()]
    
    np.random.seed(42)
    jitter = np.random.uniform(-0.15, 0.15, size=len(df_acc))
    df_acc_jittered = df_acc.copy()
    df_acc_jittered['Año_Jittered'] = df_acc_jittered['Año'] + jitter
    
    sns.scatterplot(
        data=df_acc_jittered,
        x='Año_Jittered',
        y='Accuracy_Num',
        hue='Metodologia_Categorizada',
        palette='Set2',
        s=90,
        edgecolor='black',
        alpha=0.9,
        ax=ax
    )
    ax.set_title('Evolución de la Exactitud (Accuracy %) de los Modelos por Año', pad=15, fontweight='bold')
    ax.set_xlabel('Año de Publicación')
    ax.set_ylabel('Exactitud (Accuracy %)')
    ax.set_ylim(50, 101.5)
    ax.set_xticks(sorted(df['Año'].dropna().unique().astype(int)))
    
    if is_standalone:
        ax.legend(title='Metodología', bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.tight_layout()
        plt.savefig('graficos/11_dispersion_accuracy.png', dpi=300, bbox_inches='tight')
        plt.close()
    else:
        ax.legend(title='Metodología', loc='upper left', fontsize=8, title_fontsize=9)

# ---- GRÁFICO 12: Tipo de Publicación (Revista vs Conferencia) ----
def plot_tipo_publicacion(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
        is_standalone = True
    else:
        is_standalone = False
        
    pub_counts = df['Tipo_Publicacion'].value_counts()
    ax.pie(pub_counts.values, labels=pub_counts.index, autopct='%1.1f%%', startangle=90, 
           colors=['#457B9D', '#E76F51', '#2A9D8F'],
           wedgeprops={'edgecolor': 'black', 'linewidth': 0.8, 'antialiased': True})
    centre_circle = plt.Circle((0,0),0.70,fc='white',edgecolor='black',linewidth=0.5)
    ax.add_artist(centre_circle)
    ax.set_title('Distribución por Tipo de Publicación', pad=20, fontweight='bold')
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/12_tipo_publicacion.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 13: Top 10 Canales de Publicación (Venues) ----
def plot_top_venues(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 6))
        is_standalone = True
    else:
        is_standalone = False
        
    df_venues = df[df['Venue'].notna()]
    venue_counts = df_venues['Venue'].value_counts().head(10)
    
    barplot13 = sns.barplot(x=venue_counts.values, y=venue_counts.index, hue=venue_counts.index, palette='plasma', edgecolor='black', linewidth=0.7, legend=False, ax=ax)
    ax.set_title('Top 10 Canales de Publicación (Journals y Conferencias)', pad=15, fontweight='bold')
    ax.set_xlabel('Cantidad de Artículos Publicados')
    ax.set_ylabel('Revista / Conferencia')
    
    for p in barplot13.patches:
        val = int(p.get_width())
        ax.annotate(f' {val}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    ax.set_xlim(0, max(venue_counts.values) + 1 if len(venue_counts) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/13_top_venues.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 14: Barras Agrupadas (Cruce 2x2: Tipo de Publicación vs Enfoque Tecnológico) ----
def plot_heatmap_cruce_2x2(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6.5))
        is_standalone = True
    else:
        is_standalone = False
        
    ct = pd.crosstab(df['Tipo_Publicacion'], df['Metodologia_2x2'])
    df_plot = ct.reset_index().melt(id_vars='Tipo_Publicacion', value_name='Cantidad', var_name='Metodología')
    
    barplot = sns.barplot(
        data=df_plot,
        x='Tipo_Publicacion',
        y='Cantidad',
        hue='Metodología',
        palette=['#457B9D', '#E76F51'],
        edgecolor='black',
        linewidth=0.7,
        ax=ax
    )
    
    ax.set_title('Distribución Científica: Tipo de Publicación vs. Enfoque Metodológico', pad=20, fontweight='bold', fontsize=12)
    ax.set_xlabel('Tipo de Publicación', fontsize=11)
    ax.set_ylabel('Cantidad de Artículos', fontsize=11)
    
    for p in barplot.patches:
        height = p.get_height()
        if not pd.isna(height) and height > 0:
            ax.annotate(f'{int(height)}', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='bottom', fontsize=11, color='black', xytext=(0, 3),
                        textcoords='offset points')
                        
    ax.set_ylim(0, max(df_plot['Cantidad']) * 1.15 if len(df_plot) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/14_heatmap_cruce_2x2.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 15: Barras Apiladas al 100% (Cruce 3x3: Período vs Metodología) ----
def plot_heatmap_cruce_3x3_temporal(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6.5))
        is_standalone = True
    else:
        is_standalone = False
        
    df_filtered = df
    ct = pd.crosstab(df_filtered['Periodo_Año'], df_filtered['Metodologia_3x3'])
    
    rows = [r for r in ['2020 - 2022', '2023 - 2024', '2025 - 2026', 'No especificado'] if r in ct.index]
    ct = ct.reindex(rows)
    
    cols = [c for c in ['Deep Learning (DL)', 'Machine Learning (ML)', 'Otras / Híbrido / Tradicional'] if c in ct.columns]
    ct = ct[cols]
    
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    
    ct_pct.plot(
        kind='bar',
        stacked=True,
        color=['#1D3557', '#E76F51', '#E9C46A'],
        edgecolor='black',
        linewidth=0.7,
        ax=ax
    )
    
    ax.set_title('Evolución Proporcional de Metodologías por Período de Tiempo', pad=20, fontweight='bold', fontsize=12)
    ax.set_xlabel('Período de Publicación', fontsize=11)
    ax.set_ylabel('Porcentaje (%)', fontsize=11)
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=0)
    
    for r_idx, row_name in enumerate(ct.index):
        cumm_pct = 0.0
        for col_name in ct.columns:
            val = ct.loc[row_name, col_name]
            pct = ct_pct.loc[row_name, col_name]
            if pct > 4:
                y_pos = cumm_pct + pct / 2.0
                ax.text(r_idx, y_pos, f'{pct:.1f}%\n({val})', ha='center', va='center', 
                        color='white' if col_name != 'Otras / Híbrido / Tradicional' else 'black', 
                        fontweight='bold', fontsize=9)
            cumm_pct += pct
            
    ax.legend(title='Metodología', loc='upper left', bbox_to_anchor=(1.02, 1) if is_standalone else (1.0, 1.0))
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/15_heatmap_cruce_3x3_temporal.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 16: Matriz de Burbujas (Cruce: Entorno de Prueba vs Metodología) ----
def plot_heatmap_cruce_3x3_entornos(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 7.5))
        is_standalone = True
    else:
        is_standalone = False
        
    df_filtered = df
    ct = pd.crosstab(df_filtered['Metodologia_3x3'], df_filtered['Entorno_3x3'])
    
    rows = [r for r in ['Deep Learning (DL)', 'Machine Learning (ML)', 'Otras / Híbrido / Tradicional'] if r in ct.index]
    cols = [c for c in ['Cámaras (CCTV)', 'Drones / UAV', 'Vía pública / Real', 'Otros', 'No especificado'] if c in ct.columns]
    ct = ct.reindex(index=rows, columns=cols)
    
    df_plot = ct.reset_index().melt(id_vars='Metodologia_3x3', value_name='Cantidad', var_name='Entorno')
    
    x_categories = cols
    y_categories = rows
    
    x_map = {cat: idx for idx, cat in enumerate(x_categories)}
    y_map = {cat: idx for idx, cat in enumerate(y_categories)}
    
    df_plot['X_num'] = df_plot['Entorno'].map(x_map)
    df_plot['Y_num'] = df_plot['Metodologia_3x3'].map(y_map)
    
    max_count = df_plot['Cantidad'].max()
    scale_factor = 2500 / max_count if max_count > 0 else 500
    
    scatter = ax.scatter(
        x=df_plot['X_num'],
        y=df_plot['Y_num'],
        s=df_plot['Cantidad'] * scale_factor,
        c=df_plot['Cantidad'],
        cmap='viridis_r',
        edgecolors='black',
        linewidths=1.0,
        alpha=0.85
    )
    
    for idx, row in df_plot.iterrows():
        val = int(row['Cantidad'])
        if val > 0:
            ax.annotate(
                f'{val}',
                (row['X_num'], row['Y_num']),
                ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if val > (max_count * 0.4) else 'black'
            )
            
    ax.set_xticks(range(len(x_categories)))
    ax.set_xticklabels(x_categories, fontsize=10)
    ax.set_yticks(range(len(y_categories)))
    ax.set_yticklabels(y_categories, fontsize=10)
    
    ax.set_xlim(-0.5, len(x_categories) - 0.5)
    ax.set_ylim(-0.5, len(y_categories) - 0.5)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, color='lightgray', zorder=0)
    ax.set_axisbelow(True)
    
    ax.set_title('Preferencia de Metodologías por Entorno Experimental (Matriz de Burbujas)', pad=20, fontweight='bold', fontsize=12)
    ax.set_xlabel('Entorno Experimental', fontsize=11, labelpad=10)
    ax.set_ylabel('Enfoque Metodológico', fontsize=11, labelpad=10)
    
    if is_standalone:
        plt.colorbar(scatter, ax=ax, label='Cantidad de Artículos', shrink=0.8)
        plt.tight_layout()
        plt.savefig('graficos/16_heatmap_cruce_3x3_entornos.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 17: Barras Agrupadas (Cruce: Plataformas de Implementación vs Metodología) ----
def plot_heatmap_cruce_implementacion_metodologia(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 7))
        is_standalone = True
    else:
        is_standalone = False
        
    df_filtered = df
    ct = pd.crosstab(df_filtered['Implementacion_Cruce'], df_filtered['Metodologia_3x3'])
    
    impl_order = ct.sum(axis=1).sort_values(ascending=False).index.tolist()
    ct = ct.reindex(impl_order)
    
    cols = [c for c in ['Deep Learning (DL)', 'Machine Learning (ML)', 'Otras / Híbrido / Tradicional'] if c in ct.columns]
    ct = ct[cols]
    
    df_plot = ct.reset_index().melt(id_vars='Implementacion_Cruce', value_name='Cantidad', var_name='Metodología')
    
    barplot = sns.barplot(
        data=df_plot,
        x='Implementacion_Cruce',
        y='Cantidad',
        hue='Metodología',
        palette=['#1D3557', '#E76F51', '#E9C46A'],
        edgecolor='black',
        linewidth=0.7,
        ax=ax
    )
    
    ax.set_title('Plataformas de Implementación vs. Enfoque Metodológico', pad=20, fontweight='bold', fontsize=12)
    ax.set_xlabel('Plataforma / Hardware de Implementación', fontsize=11)
    ax.set_ylabel('Cantidad de Artículos', fontsize=11)
    
    for p in barplot.patches:
        height = p.get_height()
        if not pd.isna(height) and height > 0:
            ax.annotate(f'{int(height)}', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 2),
                        textcoords='offset points')
                        
    ax.set_ylim(0, max(df_plot['Cantidad']) * 1.15 if len(df_plot) > 0 else 10)
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/17_cruce_implementacion_metodologia.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 18: Barras Apiladas al 100% (Cruce: Tipo de Dataset vs Período Temporal) ----
def plot_heatmap_cruce_tipo_dataset_temporal(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6.5))
        is_standalone = True
    else:
        is_standalone = False
        
    df_filtered = df
    ct = pd.crosstab(df_filtered['Periodo_Año'], df_filtered['Dataset_Tipo'])
    
    rows = [r for r in ['2020 - 2022', '2023 - 2024', '2025 - 2026', 'No especificado'] if r in ct.index]
    ct = ct.reindex(rows)
    
    cols = [c for c in ['Imágenes', 'Video / Frames', 'Sujetos Humanos', 'Otros / Especificado', 'No especificado'] if c in ct.columns]
    for c in ct.columns:
        if c not in cols:
            cols.append(c)
    ct = ct[cols]
    
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    
    colors = ['#2A9D8F', '#E76F51', '#457B9D', '#E9C46A', '#8D99AE', '#BDBDBD']
    ct_pct.plot(
        kind='bar',
        stacked=True,
        color=colors[:len(cols)],
        edgecolor='black',
        linewidth=0.7,
        ax=ax
    )
    
    ax.set_title('Evolución Proporcional del Tipo de Dataset por Período', pad=20, fontweight='bold', fontsize=12)
    ax.set_xlabel('Período de Publicación', fontsize=11)
    ax.set_ylabel('Porcentaje (%)', fontsize=11)
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=0)
    
    for r_idx, row_name in enumerate(ct.index):
        cumm_pct = 0.0
        for col_name in ct.columns:
            val = ct.loc[row_name, col_name]
            pct = ct_pct.loc[row_name, col_name]
            if pct > 4:
                y_pos = cumm_pct + pct / 2.0
                ax.text(r_idx, y_pos, f'{pct:.1f}%\n({val})', ha='center', va='center', 
                        color='white' if col_name not in ['Otros / Especificado', 'No especificado'] else 'black', 
                        fontweight='bold', fontsize=9)
            cumm_pct += pct
            
    ax.legend(title='Tipo de Dataset', loc='upper left', bbox_to_anchor=(1.02, 1) if is_standalone else (1.0, 1.0))
    
    if is_standalone:
        plt.tight_layout()
        plt.savefig('graficos/18_cruce_tipo_dataset_temporal.png', dpi=300, bbox_inches='tight')
        plt.close()

# ---- GRÁFICO 19: Matriz de Burbujas (Cruce: Limitaciones vs Metodología) ----
def plot_heatmap_cruce_limitaciones_metodologia(df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 7.5))
        is_standalone = True
    else:
        is_standalone = False
        
    df_filtered = df
    ct = pd.crosstab(df_filtered['Metodologia_3x3'], df_filtered['Limitacion_Cruce'])
    
    rows = [r for r in ['Deep Learning (DL)', 'Machine Learning (ML)', 'Otras / Híbrido / Tradicional'] if r in ct.index]
    lim_cols = ct.sum(axis=0).sort_values(ascending=False).index.tolist()
    ct = ct.reindex(index=rows, columns=lim_cols)
    
    df_plot = ct.reset_index().melt(id_vars='Metodologia_3x3', value_name='Cantidad', var_name='Limitacion')
    
    x_categories = lim_cols
    y_categories = rows
    
    x_map = {cat: idx for idx, cat in enumerate(x_categories)}
    y_map = {cat: idx for idx, cat in enumerate(y_categories)}
    
    df_plot['X_num'] = df_plot['Limitacion'].map(x_map)
    df_plot['Y_num'] = df_plot['Metodologia_3x3'].map(y_map)
    
    max_count = df_plot['Cantidad'].max()
    scale_factor = 2500 / max_count if max_count > 0 else 500
    
    scatter = ax.scatter(
        x=df_plot['X_num'],
        y=df_plot['Y_num'],
        s=df_plot['Cantidad'] * scale_factor,
        c=df_plot['Cantidad'],
        cmap='plasma_r',
        edgecolors='black',
        linewidths=1.0,
        alpha=0.85
    )
    
    for idx, row in df_plot.iterrows():
        val = int(row['Cantidad'])
        if val > 0:
            ax.annotate(
                f'{val}',
                (row['X_num'], row['Y_num']),
                ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if val > (max_count * 0.4) else 'black'
            )
            
    ax.set_xticks(range(len(x_categories)))
    ax.set_xticklabels(x_categories, fontsize=9, rotation=15, ha='right')
    ax.set_yticks(range(len(y_categories)))
    ax.set_yticklabels(y_categories, fontsize=10)
    
    ax.set_xlim(-0.5, len(x_categories) - 0.5)
    ax.set_ylim(-0.5, len(y_categories) - 0.5)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, color='lightgray', zorder=0)
    ax.set_axisbelow(True)
    
    ax.set_title('Incidencia de Limitaciones según Enfoque Metodológico (Matriz de Burbujas)', pad=25, fontweight='bold', fontsize=12)
    ax.set_xlabel('Principales Limitaciones Reportadas', fontsize=11, labelpad=10)
    ax.set_ylabel('Enfoque Metodológico', fontsize=11, labelpad=10)
    
    if is_standalone:
        plt.colorbar(scatter, ax=ax, label='Cantidad de Artículos', shrink=0.8)
        plt.tight_layout()
        plt.savefig('graficos/19_cruce_limitaciones_metodologia.png', dpi=300, bbox_inches='tight')
        plt.close()


# ==============================================================================


### Gráfico 1: Evolución Temporal de las Publicaciones por Año

In [ ]:
plot_publicaciones_por_año(df)
plt.show()

### Gráfico 2: Distribución de Metodologías (Filtrado)

In [ ]:
plot_distribucion_metodologias(df)
plt.show()

### Gráfico 3: Entornos Experimentales de Evaluación (Filtrado)

In [ ]:
plot_entornos_experimentales(df)
plt.show()

### Gráfico 4: Plataformas de Implementación de Software (Filtrado)

In [ ]:
plot_implementacion_software(df)
plt.show()

### Gráfico 5: Limitaciones y Falsos Positivos (Filtrado)

In [ ]:
plot_limitaciones(df)
plt.show()

### Gráfico 6: Métricas de Rendimiento Reportadas

In [ ]:
plot_metricas_rendimiento(df)
plt.show()

### Gráfico 7: Distribución por Tipo de Dataset (Filtrado)

In [ ]:
plot_tipo_dataset(df)
plt.show()

### Gráfico 8: Evolución Histórica de Metodologías (Año vs Metodología) (Filtrado)

In [ ]:
plot_evolucion_metodologias(df)
plt.show()

### Gráfico 9: Popularidad de las Versiones de YOLO

In [ ]:
plot_popularidad_yolo(df)
plt.show()

### Gráfico 10: Mapa de Calor (Heatmap) de Entorno Experimental vs Implementación

In [ ]:
plot_heatmap_relacion(df)
plt.show()

### Gráfico 11: Evolución de la Exactitud (Accuracy %) por Año (Dispersión con Jitter)

In [ ]:
plot_dispersion_accuracy(df)
plt.show()

### Gráfico 12: Tipo de Publicación (Revista vs Conferencia)

In [ ]:
plot_tipo_publicacion(df)
plt.show()

### Gráfico 13: Top 10 Canales de Publicación (Venues) (Filtrado)

In [ ]:
plot_top_venues(df)
plt.show()

## 7. Gráficos de Contingencia, Estadísticos y Cruce de Datos (Formatos Alternativos)

A continuación se presentan los gráficos combinados y estadísticos rediseñados (Barras Agrupadas, Barras Apiladas al 100%, Matrices de Burbujas, Boxplots y Línea de Tendencia).

### Gráfico 14: Cruce 2x2 de Tipo de Publicación vs. Enfoque Tecnológico (Barras Agrupadas)

In [ ]:
plot_heatmap_cruce_2x2(df)
plt.show()

### Gráfico 15: Cruce 3x3 Temporal de Período vs. Metodología (Barras Apiladas al 100%)

In [ ]:
plot_heatmap_cruce_3x3_temporal(df)
plt.show()

### Gráfico 16: Cruce 3x3 de Entorno de Prueba vs. Enfoque Metodológico (Matriz de Burbujas)

In [ ]:
plot_heatmap_cruce_3x3_entornos(df)
plt.show()

### Gráfico 17: Cruce de Plataformas de Implementación vs. Enfoque Metodológico (Barras Agrupadas)

In [ ]:
plot_heatmap_cruce_implementacion_metodologia(df)
plt.show()

### Gráfico 18: Cruce Temporal de Tipo de Dataset por Períodos (Barras Apiladas al 100%)

In [ ]:
plot_heatmap_cruce_tipo_dataset_temporal(df)
plt.show()

### Gráfico 19: Incidencia de Limitaciones Principales según Enfoque Metodológico (Matriz de Burbujas)

In [ ]:
plot_heatmap_cruce_limitaciones_metodologia(df)
plt.show()

### Gráfico 20: Comparativa de Exactitud (Accuracy %) por Enfoque Metodológico (Boxplot)

In [ ]:
plot_boxplot_accuracy_metodologia(df)
plt.show()

### Gráfico 21: Comparativa de Exactitud (Accuracy %) por Tipo de Publicación (Boxplot)

In [ ]:
plot_boxplot_accuracy_tipo_publicacion(df)
plt.show()

### Gráfico 22: Relación entre Limitaciones e Inconvenientes vs. Entorno Experimental (Matriz de Burbujas)

In [ ]:
plot_heatmap_cruce_limitaciones_entorno(df)
plt.show()

### Gráfico 23: Tendencia Temporal del Volumen de Publicaciones (Gráfico de Líneas)

In [ ]:
plot_linea_publicaciones_año(df)
plt.show()